In [22]:
import pandas as pd
import pandas_datareader.data as web
import datetime as dt
import os



os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

start = dt.datetime(1990, 1, 1)
end = dt.datetime.today()



print("Downloading Brent oil price from FRED...")
oil = web.DataReader("DCOILBRENTEU", "fred", start, end)
oil.rename(columns={"DCOILBRENTEU": "oil_price"}, inplace=True)
oil.to_csv("../data/raw/oil_price_fred.csv")



print("Downloading CPI (inflation) from FRED...")
cpi = web.DataReader("CPIAUCSL", "fred", start, end)
cpi.rename(columns={"CPIAUCSL": "cpi"}, inplace=True)
cpi.to_csv("../data/raw/cpi_fred.csv")



cons = pd.read_csv("../data/raw/consumption_worldbank.csv")
cons = cons[cons["Country Code"] == "DEU"]
year_cols = [col for col in cons.columns if col.isdigit()]
cons = cons[year_cols].T
cons.columns = ["consumption"]
cons.index = pd.to_datetime(cons.index)
cons = cons.asfreq("YE")
cons.to_csv("../data/raw/consumption_worldbank_clean.csv")



gdp = pd.read_csv("../data/raw/gdp_worldbank.csv", skiprows=4)
gdp = gdp[gdp["Country Code"] == "DEU"]
year_cols = [col for col in gdp.columns if col.isdigit()]
gdp = gdp[year_cols].T
gdp.columns = ["gdp"]
gdp.index = pd.to_datetime(gdp.index)
gdp = gdp.asfreq("YE")
gdp.to_csv("../data/raw/gdp_worldbank_clean.csv")

print("\nAll datasets downloaded and saved successfully!")



oil = pd.read_csv("../data/raw/oil_price_fred.csv", index_col=0, parse_dates=True)
cpi = pd.read_csv("../data/raw/cpi_fred.csv", index_col=0, parse_dates=True)
cons = pd.read_csv("../data/raw/consumption_worldbank_clean.csv", index_col=0, parse_dates=True)
gdp = pd.read_csv("../data/raw/gdp_worldbank_clean.csv", index_col=0, parse_dates=True)



cons = cons[cons.index >= "1990-01-01"]
gdp = gdp[gdp.index >= "1990-01-01"]



print("Resampling all datasets to monthly...")

oil = oil.resample("M").mean()
cpi = cpi.resample("M").ffill()
cons = cons.resample("M").ffill()
gdp = gdp.resample("M").ffill()



print("Merging...")
df = oil.join(cpi, how="outer")
df = df.join(cons, how="left")
df = df.join(gdp, how="left")

df = df.sort_index().ffill()

print("After merge:", df.shape)
display(df.head())



df = df[df.index >= "1991-01-01"]



df["oil_return"] = df["oil_price"].pct_change(fill_method=None)
df["cpi_change"] = df["cpi"].pct_change(fill_method=None)
df["cons_growth"] = df["consumption"].pct_change(fill_method=None)
df["gdp_growth"] = df["gdp"].pct_change(fill_method=None)

df["target_cpi_next_month"] = df["cpi"].shift(-1)

df.dropna(inplace=True)

print("Final dataset shape:", df.shape)

df.to_csv("../data/processed/final_dataset.csv")
print("Saved: data/processed/final_dataset.csv")

df.head()



All datasets downloaded and saved successfully!
Resampling all datasets to monthly...
Merging...
After merge: (435, 4)


/tmp/ipykernel_7690/2702960981.py:84: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  oil = oil.resample("M").mean()
/tmp/ipykernel_7690/2702960981.py:85: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  cpi = cpi.resample("M").ffill()
/tmp/ipykernel_7690/2702960981.py:86: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  cons = cons.resample("M").ffill()
/tmp/ipykernel_7690/2702960981.py:87: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  gdp = gdp.resample("M").ffill()


,oil_price,cpi,consumption,gdp
DATE,,,,
1990-01-31,21.251818,127.5,NaN,NaN
1990-02-28,19.813500,128.0,NaN,NaN
1990-03-31,18.387273,128.6,NaN,NaN
1990-04-30,16.612105,128.9,NaN,NaN
1990-05-31,16.352273,129.1,NaN,NaN


Final dataset shape: (0, 9)
Saved: data/processed/final_dataset.csv


,oil_price,cpi,consumption,gdp,oil_return,cpi_change,cons_growth,gdp_growth,target_cpi_next_month
DATE,,,,,,,,,
